In [ ]:
from __future__ import annotations

import numpy as np
from puwr import tauint
from dataclasses import dataclass
import time
import pyerrors as pe
#import pandas as pd

def WrapAngles(Theta: np.ndarray) -> np.ndarray:
    #Project angles back to interval [-pi, pi)
    return (Theta + np.pi) % (2.0 * np.pi) - np.pi

@dataclass 
class Lattice2D: # Defining Lattice parameters Nt-by-Nx
    Nt: int
    Nx: int

    @property
    def Shape(self) -> tuple[int, int]:
        return self.Nt, self.Nx

    @property
    def Volume(self) -> int:  
        return self.Nt * self.Nx

@dataclass
class U1Lattice2D:
    Lattice: Lattice2D # Calling the Lattice by its Geometry as defined above (NtxNx)
    Theta:   np.ndarray # Introducing Theta as an array

    @classmethod
    def ColdStart(cls, Lattice: Lattice2D) -> "U1Lattice2D": #Defining the ColdStart
        Theta = np.zeros((Lattice.Nt, Lattice.Nx, 2), dtype = np.float64)
        return cls(Lattice, Theta)

    @classmethod 
    def HotStart(cls, Lattice: Lattice2D, prng: np.random.Generator) -> "U1Lattice2D": #Defining HotStart
        Theta = prng.uniform(low = -np.pi, high = np.pi, size = (Lattice.Nt, Lattice.Nx, 2))
        return cls(Lattice, Theta)

    def Copy(self) -> "U1Lattice2D":
        return U1Lattice2D(self.Lattice, self.Theta.copy())

def StapleSum(Field: U1Lattice2D, mu: int) -> np.ndarray:
    if mu not in (0, 1):
        raise ValueError("Directional index mu must be either 0 or 1!")
    nu = 1 - mu
    forward_staple  = (Field.Theta[:, :, nu]
                    +  np.roll(Field.Theta[:, :, mu], shift = -1, axis = nu)
                    -  np.roll(Field.Theta[:, :, nu], shift = -1, axis = mu))
    backward_staple = (-np.roll(Field.Theta[:, :, nu], shift = 1, axis = nu)
                    +   np.roll(Field.Theta[:, :, mu], shift = 1, axis = nu)
                    +   np.roll(np.roll(Field.Theta[:, :, nu], shift = 1, axis = nu), shift = -1, axis = mu))
    return np.exp(1j * forward_staple) + np.exp(1j * backward_staple)


def MetropolisSweep(Field: U1Lattice2D, beta: float, stepsize: float, prng: np.random.Generator, n_hits: int) -> float:
    Nt, Nx         = Field.Lattice.Shape
    grid_t, grid_x = np.indices((Nt, Nx))
    parity         = (grid_t + grid_x) % 2
# in checkerboard (parity) update, we're only updating the independent sites
# when updating site x,its neighors don't change, so the staple at x also doesn't change
# to implement the Multi hit Metropolis, we want to propose multiple updates to a site in a row without having to recalculate the staple, the motivation is that it will help improve acceptance rate and decorrelates lattice faster

    accepted_proposals = 0
    total_proposals    = 0

    # Restrict both Nx and Nt to even integers
    if Field.Lattice.Nt % 2 != 0 or Field.Lattice.Nx % 2 != 0:
        raise ValueError("MetropolisSweep requires both Nt and Nx to be even!")

    for mu in (0, 1):
        for current_parity in (0, 1):
            parity_mask    = (parity == current_parity)
            
            ## STAPLE AND THETA_OLD FOR MULT-HIT METROPOLIS
            # Computing staple once outside the multi hit loop
            staple = StapleSum(Field, mu)
            
            # extract the starting state for this parity sweep
            theta_old = Field.Theta[:, :, mu].copy()
    
            # MULTI-HIT METROPOLIS 
            for hit in range(n_hits): 
                # proposing new angles based on the current theta_old
                theta_proposal = WrapAngles(theta_old + prng.uniform(low = -stepsize, high = stepsize, size = (Nt, Nx)))

                # Compute action difference using same staple
                DeltaS = -beta * np.real((np.exp(1j * theta_proposal) - np.exp(1j * theta_old)) * np.conjugate(staple))

                # accept/reject step
                accept_mask = parity_mask & ((DeltaS <= 0.0) | (prng.uniform(low = 0.0, high = 1.0, size = (Nt, Nx)) < np.exp(-DeltaS)))

                # Updating theta_old with the accepted proposals so the next hit builds on it
                theta_old = np.where(accept_mask, theta_proposal, theta_old)

                # Tally the acceptances
                accepted_proposals += np.count_nonzero(accept_mask)
                total_proposals    += np.count_nonzero(parity_mask)

            #After all hits are done, we write the finalized values back to the Field
            Field.Theta[:, :, mu] = theta_old

    # Return overall acceptance rate across all hits
    return accepted_proposals / total_proposals
    
def OverrelaxationSweep(Field: U1Lattice2D) -> None:
    Nt, Nx         = Field.Lattice.Shape
    grid_t, grid_x = np.indices((Nt, Nx))
    parity         = (grid_t + grid_x) % 2
    
    if Field.Lattice.Nt % 2!= 0 or Field.Lattice.Nx % 2 != 0:
        raise ValueError("OverrelaxationSweep rewuires both Nt and Nx to be even!")
        
    for mu in (0,1):
        for current_parity in (0,1):
            parity_mask = (parity == current_parity)
            
            #This is the complex staple
            staple = StapleSum(Field, mu)
            
            # Extracting the staple phase phi
            staple_phase = np.angle(staple)
            
            # Extracting old theta
            theta_old = Field.Theta[:, :, mu]
            
            # Applying the deterministic ORA equation: phi' = 2*phi  - theta
            theta_proposal = WrapAngles(2 * staple_phase -theta_old)
            
            # Updating only the sites in the current checkerboard parity
            Field.Theta[:, :, mu] = np.where(parity_mask, theta_proposal, theta_old)
            
def InstantonUpdateSweep(Field: U1Lattice2D, beta: float, prng: np.random.Generator) -> float:
    Nt, Nx = Field.Lattice.Shape
    grid_t, grid_x = np.indices((Nt, Nx))
    
    #randomly selecting an instanton charge Qj = pm 1
    Qj = prng.choice([-1.0, 1.0])
    
    #Building background instanton field modifications
    delta_theta = np.zeros((Nt, Nx, 2), dtype=np.float64)   
    #mu = 0 (temporal background) -2 * pi * x * Qj / (Nx * Nt) (like eq )
    delta_theta[:, :, 0] = -2.0 * np.pi * grid_x * Qj / (Nx * Nt)
    
    #mu = 1 (spatial background): 2 * pi * t * Qj / Nt only at boundary x = Nx - 1
    delta_theta[:, -1, 1] = 2.0 * np.pi * grid_t[:, -1] * Qj / Nt
    
    #constructing proposal configuration
    theta_proposal = WrapAngles(Field.Theta + delta_theta)
    
    #Global Metropolis Accept/Reject Step
    S_old = Action(Field, beta)
    proposed_field = U1Lattice2D(Field.Lattice, theta_proposal)
    
    S_new = Action(proposed_field, beta)
    delta_S = S_new - S_old
    
    if delta_S <= 0.0 or prng.uniform(0.0, 1.0) < np.exp(-delta_S):
        Field.Theta = theta_proposal
        return 1.0  #Accepted
    
    return 0.0  #Rejected

def HeatBathSweep(Field: U1Lattice2D, beta: float, prng: np.random.Generator) -> None:
                 
    Nt, Nx, = Field.Lattice.Shape
    
    #creating an array with the x and t-coordinate of every lattice site
    grid_t, grid_x = np.indices((Nt,Nx))
    #we use the same checkerboard structure as we did in the MetropolisSweep (only updating one independent parity at a time)
    parity = (grid_t + grid_x) % 2
    
    if Nt % 2 != 0 or Nx % 2!= 0:
        raise ValueError("HeatBathSweep requires both Nt and Nx to be even!")
        
    for mu in (0,1):
        
        for current_parity in (0,1):
        
            parity_mask = (parity == current_parity)
            #Calculating the staple (the StapleSum already returns the complex staple so we don't need a new staple function)
            staple = StapleSum(Field, mu)
        
            #we extract angle and magnitude of staple
            staple_phase = np.angle(staple)
            staple_magnitude = np.abs(staple)
        
            #defining von Mises concentration
            kappa = beta * staple_magnitude
        
            #we want to sample the new link angle
            #NumPy's Generator.vonmises(mu,kappa) uses mu as model/location angle and kappa as the concentration parameter,
            #and the function return angles in [-pi,pi]
            theta_new = prng.vonmises(
                mu = staple_phase,
                kappa = kappa
            )
        
            Field.Theta[:, :, mu] = np.where(
                parity_mask,
                theta_new,
                Field.Theta[:, :, mu]
            )   
                       
# Observables

def PlaquetteAngles(Field: U1Lattice2D) -> np.ndarray:
    # Plaq = exp(i * arg) 
    # arg = theta_t + np.roll(theta_x, shift = -1, axis = 0) - np.roll(theta_t, shift = -1, axis = 1) - theta_x
    theta_plaq = (Field.Theta[:, :, 0]
               +  np.roll(Field.Theta[:, :, 1], shift = -1, axis = 0)
               -  np.roll(Field.Theta[:, :, 0], shift = -1, axis = 1)
               -  Field.Theta[:, :, 1])
    return WrapAngles(theta_plaq)

def Action(Field: U1Lattice2D, beta: float) -> float:
    return float(beta * np.sum(1.0 - np.cos(PlaquetteAngles(Field))))

def AveragePlaquette(Field: U1Lattice2D) -> float:
    return float(np.mean(np.cos(PlaquetteAngles(Field))))

# General Wilson Loop NbyN
def WilsonLoop(Field: U1Lattice2D, Lt: int, Lx: int) -> float:
    #Initializing array to hold the accumulated phase for loops at each site
    loop_phase= np.zeros(Field.Lattice.Shape, dtype=np.float64)
    
    #bottom edge: going +x for Lx steps, starting at time t
    for i in range(Lx):
        loop_phase += np.roll(Field.Theta[:,:,1], shift=-i, axis =1)

    #right: going +t for Lt steps, starting at space x+Lx 
    for j in range(Lt):
        loop_phase += np.roll(Field.Theta[:, :, 0],shift =(-j,-Lx), axis = (0,1))
        
    #top: going -x for Lx steps (backward), starting at time t+Lt
    for i in range(Lx):
        loop_phase -= np.roll(Field.Theta[:, :, 1], shift = (-Lt, -i), axis = (0, 1))
        
    #left edge: going -t for Lt steps, starting at space x
    for j in range(Lt):
        loop_phase -= np.roll(Field.Theta[:, :, 0], shift = -j, axis = 0)
    #Wilson loop is the real part of: Re[exp(i * phase)] = cos(phase)
    return float(np.mean(np.cos(loop_phase)))

def TopologicalCharge(Field: U1Lattice2D) -> float:
    return float(1.0 / (2.0 * np.pi) * np.imag(np.sum(np.log(np.exp(1.0j * PlaquetteAngles(Field))))))

# Block bootstrap
def BlockBootstrapError(data: np.ndarray, tau_int: float, n_boot: int = 200, prng: np.random.Generator = None) -> float:
    if prng is None:
        prng = np.random.default_rng()
        
    N = len(data)
    # The block size must be an integer (at least 1)
    block_size = max(1, int(np.ceil(2.0 * tau_int)))
    n_blocks = N // block_size
    
    if n_blocks == 0:
        raise ValueError("Data length is too short for the calculated block size.")

    # Truncating the data slightly so it divides into blocks
    trunct_data = data[:n_blocks * block_size].reshape((n_blocks, block_size))
    
    # Generating random block indices for all bootstrap samples (at once)
    rndm_indices = prng.integers(0, n_blocks, size=(n_boot, n_blocks)) #matrix of shape (n_boot, n_blocks)
    
    boot_means = np.mean(trunct_data[rndm_indices], axis=(1, 2))
    
    # Std of the bootstrap
    return float(np.std(boot_means))


if __name__ == "__main__": #What should we call this later?
    rng = np.random.default_rng()

    Nt   = 32
    Nx   = 32
    beta = 12.8
    
    stepsize    = 0
    n_updates   = 20000 #Measurement updates
    n_thermal   = 1000 #number of sweeps to reach equilibrium
    n_hits      = 0 # number of hits for the MetropolisSweep
    n_ORA       = 0 #I want to increase this to test the efficiency of ORA
    n_instanton = 0 # Number of global Instanton updates per loop
    S           = 2 #Controls how the Pyerrors package chooses the summation window W (the default is 2 anyways)
    n_HB        = 1 #HB sweeps?
    
    print(f"{Nt}x{Nx} lattice at beta = {beta}")
    
    
    # Initializing the lattice geometry; I started with a Cold Start
    lattice = Lattice2D(Nt,Nx)
    field   = U1Lattice2D.ColdStart(lattice) # cold start
    #field  = U1Lattice2D.HotStart(lattice, rng) #this is for the hot start
    
    # Perform thermalization updates
    print(f"Starting thermalization ({n_thermal} sweeps)")
    
    for _ in range(n_thermal):
    # Calling the sweep to update the field, but I haven't saved the observables yet
        #MetropolisSweep(field,beta,stepsize,rng, n_hits)
        
        #For HB Algorithm
        for _ in range(n_HB):
            HeatBathSweep(field, beta, rng)
        
        
        for _ in range(n_ORA):
            OverrelaxationSweep(field)
            
        for _ in range(n_instanton):
            InstantonUpdateSweep(field, beta, rng)
    print("Thermalization complete")
    

    # Allocate arrays for each observable measurement
    P_measurements        = np.zeros(n_updates, dtype = np.float64)
    Q_measurements        = np.zeros(n_updates, dtype = np.float64)
    acc_rates             = np.zeros(n_updates, dtype = np.float64)
    inst_acc_rates        = np.zeros(n_updates, dtype = np.float64)
    Wnxn_measurements     = np.zeros(n_updates, dtype = np.float64)
    W2x2_measurements     = np.zeros(n_updates, dtype = np.float64)
    W4x4_measurements     = np.zeros(n_updates, dtype = np.float64)
    W8x8_measurements     = np.zeros(n_updates, dtype = np.float64)

    
    print(f"\nStarting main measurement loop ({n_updates} sweeps)")
    t0 = time.time()
    
    
    for i in range(n_updates): # running the main simulations, it looks much more efficient than our MATLAB script
        # updating the lattice and recording the acceptance rate
        #acc_rates[i]= MetropolisSweep(field, beta, stepsize, rng, n_hits)
        
        #for the HB:
        for _ in range(n_HB):
            HeatBathSweep(field, beta, rng)
        
        for _ in range(n_ORA):
            OverrelaxationSweep(field)
        
        inst_acc = 0.0
        for _ in range(n_instanton):
            inst_acc += InstantonUpdateSweep(field, beta, rng)
        if n_instanton > 0:
            inst_acc_rates[i] = inst_acc / n_instanton
        
        # measure and store observables
        P_measurements[i] = AveragePlaquette(field)
        Q_measurements[i] = TopologicalCharge(field)
        
        Wnxn_measurements[i] = WilsonLoop(field, 1, 1)
        W2x2_measurements[i] = WilsonLoop(field, 2, 2)
        W4x4_measurements[i] = WilsonLoop(field, 4, 4)
        W8x8_measurements[i] = WilsonLoop(field, 8, 8)
        
            
    t1 = time.time()
    total_time = t1 - t0
    print(f"\nMeasurement loop complete ({total_time} seconds)")
    print("Measurement loop complete")
    
    # Print mean and naive error estimate
    meanP_naive = np.mean(P_measurements)
    errP_naive  = np.std(P_measurements)/np.sqrt(n_updates)
    meanQ_naive = np.mean(Q_measurements)
    errQ_naive  = np.std(Q_measurements)/np.sqrt(n_updates)
    
    # Computing topological charge susceptibility
    V          = Nt*Nx #Volume
    Q_susc     = np.mean(Q_measurements**2) /V #Q susceptibility
    Q_susc_err = np.std(Q_measurements**2)/(np.sqrt(n_updates)*V) #Naive error estimate 
    
    mean_acc      = np.mean(acc_rates) #average acceptance rate over all sweeps
    mean_inst_acc = np.mean(inst_acc_rates) if n_instanton > 0 else 0.0
    mean_Wnxn     = np.mean(Wnxn_measurements)
    err_Wnxn      = np.std(Wnxn_measurements) / np.sqrt(n_updates)
    
    mean_W2x2 = np.mean(W2x2_measurements)
    err_W2x2  = np.std(W2x2_measurements) / np.sqrt(n_updates)
    
    mean_W4x4 = np.mean(W4x4_measurements)
    err_W4x4  = np.std(W4x4_measurements) / np.sqrt(n_updates)
    
    mean_W8x8 = np.mean(W8x8_measurements)
    err_W8x8  = np.std(W8x8_measurements) / np.sqrt(n_updates)
    
    # Calculating integrated autocorrelation time for Plaquette
    meanP_uwerr, errP_uwerr, tauP_uwerr, dtauP_uwerr = tauint([[P_measurements]], lambda x: x)
    print(f"\nAutocorrelation time tau_int for wnxn and Q")
    print(f"Plaquette tau_int: {tauP_uwerr:.2f}")
    
    # calculating tau_int for topological charge
    meanQ_uwerr, errQ_uwerr, tauQ_uwerr, dtauQ_uwerr = tauint([[Q_measurements]], lambda x: x)
    print(f"Topological Charge tau_int: {tauQ_uwerr:.2f}")
    # Applying Gamma method to Q^2
    meanQS_uwerr, errQS_uwerr, tauQS_uwerr, dtauQS_uwerr = tauint([[Q_measurements**2]], lambda x: x)
    QG_susc = meanQS_uwerr/V
    QG_err  = errQS_uwerr/V
    print(f"Topological Charge Susceptibility= {QG_susc:.2f} ± {QG_err}")
    
    meanWnxn_uwerr, errWnxn_uwerr, tauWnxn_uwerr, dtauWnxn_uwerr = tauint([[Wnxn_measurements]], lambda x: x)
    print(f"W1x1 tau_int: {tauWnxn_uwerr:.2f}")
    
    meanW2x2_uwerr, errW2x2_uwerr, tauW2x2_uwerr, dtauW2x2_uwerr = tauint([[W2x2_measurements]], lambda x: x)
    print(f"W2x2 tau_int: {tauW2x2_uwerr:.2f}")
    
    meanW4x4_uwerr, errW4x4_uwerr, tauW4x4_uwerr, dtauW4x4_uwerr = tauint([[W4x4_measurements]], lambda x: x)
    print(f"W4x4 tau_int: {tauW4x4_uwerr:.2f}")
    
    meanW8x8_uwerr, errW8x8_uwerr, tauW8x8_uwerr, dtauW8x8_uwerr = tauint([[W8x8_measurements]], lambda x: x)
    print(f"W8x8 tau_int: {tauW8x8_uwerr:.2f}")
    
    ensemble_name = (f"L{Nt}x{Nx}_B{beta}_Instanton{n_instanton}_ORA{n_ORA}_MHMA{n_hits}_HB{n_HB}")
    
    #Define observable (Plaquette)
    P_pe = pe.Obs([P_measurements],[ensemble_name])
    #gamma method analyzes entire MC history stored in an Obs. This will do the error propagation
    
    #Topological Charge
    Q_pe = pe.Obs([Q_measurements],[ensemble_name])
    
    #Topological charge susceptibility
    QSusc_pe = pe.Obs([Q_measurements**2/V],[ensemble_name])
    
    #Wilson loops (Wnxn is W1x1 in this case bc I set n=1)
    Wnxn_pe = pe.Obs([Wnxn_measurements],[ensemble_name])
    
    W2x2_pe = pe.Obs([W2x2_measurements],[ensemble_name])
    
    W4x4_pe = pe.Obs([W4x4_measurements],[ensemble_name])
    
    W8x8_pe = pe.Obs([W8x8_measurements],[ensemble_name])
    
    #now we run the Gamma method once for each observable (Gamma from pyerrors library)
    for obs in [P_pe, Q_pe, QSusc_pe, Wnxn_pe, W2x2_pe, W4x4_pe, W8x8_pe]:
        obs.gamma_method(S=S)
        
    def get_pyerrors_results(obs, ensemble_name):
        return {
            "mean": float(obs.value),
            "error": float(obs.dvalue),
            "tau": float(obs.e_tauint[ensemble_name]),
            "dtau": float(obs.e_dtauint[ensemble_name])
        }
        
    P_pe_results  = get_pyerrors_results(P_pe,ensemble_name)
    Q_pe_results  = get_pyerrors_results(Q_pe,ensemble_name)
    Wn_pe_results = get_pyerrors_results(Wnxn_pe,ensemble_name)
    W2_pe_results = get_pyerrors_results(W2x2_pe,ensemble_name)
    W4_pe_results = get_pyerrors_results(W4x4_pe,ensemble_name)
    W8_pe_results = get_pyerrors_results(W8x8_pe,ensemble_name)
    QS_pe_results = get_pyerrors_results(QSusc_pe,ensemble_name)
    
    print("\nPyerrors plaquette analysis:")
    P_pe.details()

    print("\nPyerrors topological susceptibility analysis:")
    QSusc_pe.details()
    
    # Printing mean and bootstrap error estimates
    meanP_boot = np.mean(P_measurements)
    errP_boot  = BlockBootstrapError(P_measurements, tauP_uwerr, n_boot=500, prng=rng)
    
    meanQ_boot = np.mean(Q_measurements)
    errQ_boot  = BlockBootstrapError(Q_measurements, tauQ_uwerr, n_boot=500, prng=rng)
    
    #Q susceptibility with bootstrapped uncertainty
    QB_susc = np.mean(Q_measurements**2) /V
    QB_err  = BlockBootstrapError(Q_measurements**2, tauQS_uwerr, n_boot=500, prng=rng) /V
    
    
    np.save(f"Q_history_{Nt}x{Nx}_B{beta}_Instanton{n_instanton}_ORA{n_ORA}_MHMA{n_hits}_HB{n_HB}.npy", Q_measurements)
    np.save(f"P_history_{Nt}x{Nx}_B{beta}_Instanton{n_instanton}_ORA{n_ORA}_MHMA{n_hits}_HB{n_HB}.npy", P_measurements)
    np.save(f"W2x2_history_{Nt}x{Nx}_B{beta}_Instanton{n_instanton}_ORA{n_ORA}_MHMA{n_hits}_HB{n_HB}.npy", W2x2_measurements) #I didn't save the W1x1 because I already saved the plaquette and I think it's unnecessary
    np.save(f"W4x4_history_{Nt}x{Nx}_B{beta}_Instanton{n_instanton}_ORA{n_ORA}_MHMA{n_hits}_HB{n_HB}.npy", W4x4_measurements)
    np.save(f"W8x8_history_{Nt}x{Nx}_B{beta}_Instanton{n_instanton}_ORA{n_ORA}_MHMA{n_hits}_HB{n_HB}.npy", W8x8_measurements)
    
    print(f"\nStep size = {stepsize}")
    print(f"n_hits = {n_hits}")
    print(f"n_ORA sweeps= {n_ORA}")
    print(f"n_Instanton sweeps = {n_instanton}")
    print(f"\nAcceptance rate= {mean_acc:.2f}")
    print(f"Instanton Acceptance rate= {mean_inst_acc:.2f}")
    
    print(f"\nQ susceptibility (Naive): {Q_susc:.6f} ± {Q_susc_err:.6f}")
    print(f"Q susceptibility (Gamma): {QG_susc} ± {QG_err}")
    print(f"Q susceptibility (Boot): {QB_susc} ± {QB_err}")

    
    print(f"\nWilson Loop (1x1) (Naive): {mean_Wnxn:.6f} ± {err_Wnxn:.6f}")
    print(f"Wilson Loop (2x2) (Naive): {mean_W2x2:.6f} ± {err_W2x2:.6f}")
    print(f"Wilson Loop (4x4) (Naive): {mean_W4x4:.6f} ± {err_W4x4:.6f}")   
    print(f"Wilson Loop (8x8) (Naive): {mean_W8x8:.6f} ± {err_W8x8:.6f}")
    
    print(f"\nFinal results for average Plaquette and Q")
    
    print(f"AvgPlaquette (Naive): {meanP_naive:.6f} ± {errP_naive}")
    print(f"AvgPlaquette (Bootstrap): {meanP_boot:.6f} ± {errP_boot:.6f}")
    print(f"AvgPlaquette (Gamma): {meanP_uwerr:.6f} ± {errP_uwerr:.6f}")
    
    print(f"\nQ (Naive): {meanQ_naive:.6f} ± {errQ_naive}")
    print(f"Q (Boot): {meanQ_boot:.6f} ± {errQ_boot:.6f}")
    print(f"Q (Gamma): {meanQ_uwerr:.6f} ± {errQ_uwerr:.6f}")  
    